In [1]:
!pip install kagglehub

  Using cached kagglehub-1.0.0-py3-none-any.whl.metadata (40 kB)
  Using cached kagglesdk-0.1.15-py3-none-any.whl.metadata (13 kB)
Using cached kagglehub-1.0.0-py3-none-any.whl (70 kB)
Using cached kagglesdk-0.1.15-py3-none-any.whl (160 kB)


In [2]:
import pandas as pd
import numpy as np
import glob
import gc
import seaborn as sns
import matplotlib.pyplot as plt

# Models
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical

# Preprocessing & Metrics
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import os
import kagglehub

In [3]:
LABEL_MAP = {
    "BENIGN": 0,
    "Botnet": 1,
    "Botnet - Attempted": 2,
    "DDoS": 3,
    "DoS GoldenEye": 4,
    "DoS GoldenEye - Attempted": 5,
    "DoS Hulk": 6,
    "DoS Hulk - Attempted": 7,
    "DoS Slowhttptest": 8,
    "DoS Slowhttptest - Attempted": 9,
    "DoS Slowloris": 10,
    "DoS Slowloris - Attempted": 11,
    "FTP-Patator": 12,
    "FTP-Patator - Attempted": 13,
    "Heartbleed": 14,
    "Infiltration": 15,
    "Infiltration - Attempted": 16,
    "Infiltration - Portscan": 17,
    "Portscan": 18,
    "SSH-Patator": 19,
    "SSH-Patator - Attempted": 20,
    "Web Attack - Brute Force": 21,
    "Web Attack - Brute Force - Attempted": 22,
    "Web Attack - SQL Injection": 23,
    "Web Attack - SQL Injection - Attempted": 24,
    "Web Attack - XSS": 25,
    "Web Attack - XSS - Attempted": 26
}

# Invert map for reporting (0 -> "BENIGN")
REVERSE_LABEL_MAP = {v: k for k, v in LABEL_MAP.items()}

IMPORTANT_COLS = [
    'Dst Port', 'Protocol', 'Flow Duration', 'Total Fwd Packet',
    'Total Bwd packets', 'Total Length of Fwd Packet', 'Total Length of Bwd Packet',
    'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean',
    'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min',
    'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s',
    'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Min', 'Fwd IAT Std',
    'Bwd IAT Std', 'Fwd PSH Flags', 'Bwd URG Flags', 'Fwd RST Flags',
    'Bwd RST Flags', 'Fwd Header Length', 'Bwd Header Length',
    'Bwd Packets/s', 'Packet Length Min', 'FIN Flag Count',
    'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count',
    'URG Flag Count', 'CWR Flag Count', 'ECE Flag Count', 'Down/Up Ratio',
    'Fwd Bytes/Bulk Avg', 'Fwd Packet/Bulk Avg', 'Fwd Bulk Rate Avg',
    'Bwd Bulk Rate Avg', 'Subflow Fwd Packets', 'Subflow Fwd Bytes',
    'Subflow Bwd Packets', 'Subflow Bwd Bytes', 'FWD Init Win Bytes',
    'Bwd Init Win Bytes', 'Fwd Act Data Pkts', 'Fwd Seg Size Min',
    'Active Mean', 'Active Std', 'Active Max', 'Active Min', 'ICMP Code',
    'ICMP Type', 'Total TCP Flow Time', 
    'Timestamp', 
    'Label'
]

def load_and_process(file_list):
    """Loads files, filters cols, and applies label map."""
    print(f"Reading {len(file_list)} files...")
    # Generator for memory efficiency
    df = pd.concat((pd.read_csv(f) for f in file_list), ignore_index=True)
    
    # Filter Columns
    existing_cols = [c for c in IMPORTANT_COLS if c in df.columns]
    df = df[existing_cols]
    
    # Handle NaNs/Inf
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.fillna(0, inplace=True)
    
    # Sort by Timestamp for Time Series
    if 'Timestamp' in df.columns:
        df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')
        df.sort_values('Timestamp', inplace=True)
        df.drop(columns=['Timestamp'], inplace=True)

    # Apply Label Map
    df['Label'] = df['Label'].map(LABEL_MAP)
    
    # Drop unknown labels
    if df['Label'].isna().any():
        print(f"Dropping {df['Label'].isna().sum()} rows with unknown labels.")
        df = df.dropna(subset=['Label'])
        
    df['Label'] = df['Label'].astype(int)
    return df

In [4]:
path = kagglehub.dataset_download("ernie55ernie/improved-cicids2017-and-csecicids2018")
search_path = os.path.join(path, "CICIDS2017_improved", "*.csv")
train_files = glob.glob(search_path)
train_files.append("Attack_Traffic_Dataset.csv") # Add the extra dataset to training
print("--- 1. Loading Combined Training Data ---")
df_combined = load_and_process(train_files)

--- 1. Loading Combined Training Data ---
Reading 6 files...


In [5]:
X = df_combined.drop(columns=['Label'])
y = df_combined['Label']

In [6]:
print("Splitting Combined Data into Train/Val...")
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

Splitting Combined Data into Train/Val...


In [7]:
X_train.shape

(1749725, 58)

In [8]:
X_val.shape

(437432, 58)

In [9]:
del df_combined, X, y
gc.collect()

0

In [10]:
print("\n--- 2. Loading External Test Data (Attack_Traffic_Dataset1.csv) ---")
df_ext = load_and_process(["Attack_Traffic_Dataset1.csv"])
X_test_ext = df_ext.drop(columns=['Label'])
y_test_ext = df_ext['Label']
del df_ext
gc.collect()


--- 2. Loading External Test Data (Attack_Traffic_Dataset1.csv) ---
Reading 1 files...


0

In [11]:
print("\nScaling Data...")
scaler = StandardScaler()
# Fit ONLY on training split
X_train_scaled = scaler.fit_transform(X_train)
# Transform Validation and External Test
X_val_scaled = scaler.transform(X_val)
X_test_ext_scaled = scaler.transform(X_test_ext)


Scaling Data...


In [12]:
results = []

def evaluate_model(name, model, X_v, y_v, X_e, y_e):
    """Helper to predict and print scores for both sets"""
    print(f"Evaluating {name}...")
    
    # Internal Validation
    pred_val = model.predict(X_v)
    acc_val = accuracy_score(y_v, pred_val)
    
    # External Test
    pred_ext = model.predict(X_e)
    acc_ext = accuracy_score(y_e, pred_ext)
    
    print(f"  -> Val Acc: {acc_val:.4f} | Ext Test Acc: {acc_ext:.4f}")
    return {"Model": name, "Val_Acc": acc_val, "Ext_Test_Acc": acc_ext}

In [15]:
print("\nCatBoost...")
cb = CatBoostClassifier(
    loss_function='MultiClass', 
    iterations=500, 
    depth=6, 
    learning_rate=0.1, 
    verbose=1,
    task_type="GPU"
)
cb.fit(X_train, y_train)
results.append(evaluate_model("CatBoost", cb, X_val, y_val, X_test_ext, y_test_ext))


CatBoost...
0:	learn: 1.1438225	total: 264ms	remaining: 2m 11s
1:	learn: 0.9577222	total: 446ms	remaining: 1m 50s
2:	learn: 0.8219897	total: 626ms	remaining: 1m 43s
3:	learn: 0.7189261	total: 808ms	remaining: 1m 40s
4:	learn: 0.6284538	total: 987ms	remaining: 1m 37s
5:	learn: 0.5544678	total: 1.17s	remaining: 1m 36s
6:	learn: 0.4923761	total: 1.36s	remaining: 1m 35s
7:	learn: 0.4420374	total: 1.54s	remaining: 1m 35s
8:	learn: 0.3972863	total: 1.74s	remaining: 1m 34s
9:	learn: 0.3557282	total: 1.92s	remaining: 1m 34s
10:	learn: 0.3216093	total: 2.11s	remaining: 1m 33s
11:	learn: 0.2914596	total: 2.3s	remaining: 1m 33s
12:	learn: 0.2642256	total: 2.48s	remaining: 1m 32s
13:	learn: 0.2408503	total: 2.67s	remaining: 1m 32s
14:	learn: 0.2193130	total: 2.85s	remaining: 1m 32s
15:	learn: 0.2005835	total: 3.04s	remaining: 1m 31s
16:	learn: 0.1836537	total: 3.22s	remaining: 1m 31s
17:	learn: 0.1686898	total: 3.4s	remaining: 1m 31s
18:	learn: 0.1549166	total: 3.6s	remaining: 1m 31s
19:	learn: 0

In [16]:
print("\nSVM (SGD)...")
svm = SGDClassifier(loss='hinge', n_jobs=3,verbose=1,max_iter = 500)
svm.fit(X_train_scaled, y_train)
results.append(evaluate_model("SVM", svm, X_val_scaled, y_val, X_test_ext_scaled, y_test_ext))


SVM (SGD)...


[Parallel(n_jobs=3)]: Using backend ThreadingBackend with 3 concurrent workers.


-- Epoch 1
-- Epoch 1
-- Epoch 1
Norm: 35.40, NNZs: 58, Bias: -114.204531, T: 1749725, Avg. loss: 0.030803
Total training time: 0.61 seconds.
-- Epoch 2
Norm: 12.13, NNZs: 58, Bias: 6.765240, T: 1749725, Avg. loss: 0.126213
Total training time: 0.68 seconds.
-- Epoch 2
Norm: 9.87, NNZs: 58, Bias: -194.722705, T: 1749725, Avg. loss: 0.057445
Total training time: 0.69 seconds.
-- Epoch 2
Norm: 33.28, NNZs: 58, Bias: -106.522539, T: 3499450, Avg. loss: 0.017771
Total training time: 1.20 seconds.
-- Epoch 3
Norm: 7.85, NNZs: 58, Bias: 4.366053, T: 3499450, Avg. loss: 0.045174
Total training time: 1.36 seconds.
-- Epoch 3
Norm: 10.28, NNZs: 58, Bias: -192.494429, T: 3499450, Avg. loss: 0.053245
Total training time: 1.37 seconds.
-- Epoch 3
Norm: 32.22, NNZs: 58, Bias: -102.178606, T: 5249175, Avg. loss: 0.016130
Total training time: 1.71 seconds.
-- Epoch 4
Norm: 6.56, NNZs: 58, Bias: 3.494488, T: 5249175, Avg. loss: 0.041223
Total training time: 2.05 seconds.
-- Epoch 4
Norm: 9.99, NNZs: 5

[Parallel(n_jobs=3)]: Done  27 out of  27 | elapsed:   49.4s finished


  -> Val Acc: 0.9729 | Ext Test Acc: 0.8555


In [17]:
print("\nRandom Forest...")
rf = RandomForestClassifier(n_estimators=100, n_jobs=3,verbose=1,max_samples=0.8)
rf.fit(X_train, y_train) # Uses unscaled X_train
# Note: RF uses unscaled data usually, but works with scaled too. Using unscaled for RF:
results.append(evaluate_model("RandomForest", rf, X_val, y_val, X_test_ext, y_test_ext))


Random Forest...


[Parallel(n_jobs=3)]: Using backend ThreadingBackend with 3 concurrent workers.
[Parallel(n_jobs=3)]: Done  44 tasks      | elapsed:  1.9min
[Parallel(n_jobs=3)]: Done 100 out of 100 | elapsed:  4.0min finished
[Parallel(n_jobs=3)]: Using backend ThreadingBackend with 3 concurrent workers.


Evaluating RandomForest...


[Parallel(n_jobs=3)]: Done  44 tasks      | elapsed:    1.8s
[Parallel(n_jobs=3)]: Done 100 out of 100 | elapsed:    3.8s finished
[Parallel(n_jobs=3)]: Using backend ThreadingBackend with 3 concurrent workers.
[Parallel(n_jobs=3)]: Done  44 tasks      | elapsed:    0.2s
[Parallel(n_jobs=3)]: Done 100 out of 100 | elapsed:    0.3s finished


  -> Val Acc: 0.9936 | Ext Test Acc: 0.7702


In [18]:
print("\nXGBoost...")
from xgboost import XGBClassifier
gb = XGBClassifier(
    tree_method="hist",     # FAST CPU
    n_estimators=200,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,           # speed + regularization
    colsample_bytree=0.8,
    n_jobs=3, # use all cores
)

gb.fit(X_train, y_train)

results.append(evaluate_model("XGBoost", gb, X_val, y_val, X_test_ext, y_test_ext))


XGBoost...
Evaluating XGBoost...
  -> Val Acc: 0.9946 | Ext Test Acc: 0.9823


In [20]:
print("\nLogistic Regression...")
lr = LogisticRegression(multi_class='multinomial', solver='sag', max_iter=200, n_jobs=3,verbose=1)
lr.fit(X_train_scaled, y_train)
results.append(evaluate_model("LogisticRegression", lr, X_val_scaled, y_val, X_test_ext_scaled, y_test_ext))


Logistic Regression...


[Parallel(n_jobs=3)]: Using backend ThreadingBackend with 3 concurrent workers.


Epoch 1, change: 1
Epoch 2, change: 0.62064335
Epoch 3, change: 0.24078562
Epoch 4, change: 0.083876673
Epoch 5, change: 0.060243162
Epoch 6, change: 0.055906076
Epoch 7, change: 0.050091364
Epoch 8, change: 0.043401758
Epoch 9, change: 0.037379451
Epoch 10, change: 0.032660369
Epoch 11, change: 0.030156644
Epoch 12, change: 0.028190911
Epoch 13, change: 0.026563524
Epoch 14, change: 0.025098121
Epoch 15, change: 0.023767335
Epoch 16, change: 0.022576822
Epoch 17, change: 0.02132054
Epoch 18, change: 0.020231928
Epoch 19, change: 0.019259519
Epoch 20, change: 0.018405116
Epoch 21, change: 0.017638524
Epoch 22, change: 0.016962915
Epoch 23, change: 0.016336106
Epoch 24, change: 0.015754284
Epoch 25, change: 0.015201299
Epoch 26, change: 0.014693747
Epoch 27, change: 0.014210834
Epoch 28, change: 0.013755537
Epoch 29, change: 0.013332951
Epoch 30, change: 0.012935958
Epoch 31, change: 0.0125604
Epoch 32, change: 0.012207458
Epoch 33, change: 0.011879379
Epoch 34, change: 0.011571468
Epoc

In [21]:
print("\n" + "="*50)
print("FINAL EVALUATION SUMMARY")
print("="*50)
df_res = pd.DataFrame(results)
print(df_res.sort_values("Ext_Test_Acc", ascending=False))


FINAL EVALUATION SUMMARY
                Model   Val_Acc  Ext_Test_Acc
0            CatBoost  0.994523      0.982346
3             XGBoost  0.994568      0.982346
1                 SVM  0.972876      0.855486
4  LogisticRegression  0.974414      0.820375
2        RandomForest  0.993615      0.770152
